# Install required packages

Installs and upgrades LangChain core, Google Gemini GenAI integration, and the requests library for building LLM-powered applications with external API access.

In [ ]:
!pip install -qU langchain-core langchain-google-genai requests

# Imports



In [ ]:
import os
import requests
from google.colab import userdata
from langchain.agents import Tool, AgentExecutor, create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI  # Gemini integration
from langchain_core.prompts import ChatPromptTemplate
from langchain import hub # Import hub to load a suitable prompt template

# Weather API tool

Fetches the current weather conditions for a specified city using the wttr.in API and returns a formatted string response.

In [ ]:
def get_weather(city: str) -> str:
    """Fetches current weather for a city"""
    response = requests.get(f"https://wttr.in/{city}?format=%C+%t")
    return response.text

# Configure Gemini API

Set up and use your Gemini API key here.

In [ ]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_NEW")  # Use your Gemini key

# Initialize Gemini LLM

Initializes a ChatGoogleGenerativeAI instance with the Gemini 1.5 Flash latest model for conversational AI tasks.

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest")

# Create tools list



In [ ]:
tools = [Tool(
    name="Weather API",
    func=get_weather,
    description="Get current weather conditions for any city"
)]

# Load a suitable prompt template from the hub

Loads the standard ReAct agent prompt template from the LangChain Hub for use in agent workflows.

In [ ]:
prompt = hub.pull("hwchase17/react")

# Create agent

Creates a LangChain ReAct agent with the specified LLM, tools, and prompt, then initializes an AgentExecutor with verbose output and enhanced error handling enabled.

In [ ]:
agent = create_react_agent(
    llm,
    tools,
    prompt # Use the loaded prompt
)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # Show reasoning process
    handle_parsing_errors=True  # Better error handling
)

# Execute query

Executes a LangChain agent to answer a weather and clothing recommendation query for Paris, with error handling and formatted output.

In [ ]:
try:
    response = agent_executor.invoke({
        "input": "What's the weather in Paris? How should I dress for this weather?"
    })
    print("\nFinal Response:", response['output'])
except Exception as e:
    print(f"Error executing agent: {str(e)}")

/usr/local/lib/python3.11/dist-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(




> Entering new AgentExecutor chain...
Thought: I need to get the current weather in Paris to determine appropriate clothing.
Action: Weather API
Action Input: ParisSunny +28°CThought: I now know the final answer.  28°C is warm, so light clothing is appropriate.
Final Answer: The weather in Paris is sunny and 28°C. You should wear light clothing, such as a t-shirt and shorts, and sunglasses.


> Finished chain.

Final Response: The weather in Paris is sunny and 28°C. You should wear light clothing, such as a t-shirt and shorts, and sunglasses.
